# 02 — Tokenization and Term Normalization

**From Clinical Case Reports to Knowledge Graphs**

This notebook walks step by step through the term-vocabulary concepts from
*Introduction to Information Retrieval*, Lecture 2 ("The term vocabulary and
postings lists"): normalization, tokenization problems (hyphens,
whitespace-free languages), accents/diacritics, case folding, stop words,
lemmatization, and stemming (Porter and its successors) — then extends the
lecture with the modern replacement for all of it: subword tokenization
(BPE / WordPiece / SentencePiece), as used by the transformer models this
activity's later steps depend on.

Every example is built **from the real `cases`/`metadata` tables** produced
by `01_data_preparation.ipynb` (run that notebook first if
`data/clinical_cases.duckdb` doesn't exist yet) — not toy sentences. Where an
algorithm is small enough to be worth seeing from the inside (Porter's rules,
byte-pair-encoding merges, forward-maximum-match segmentation), it's
implemented by hand and shown step by step, the same way
`information-retrieval/basics/cabot/ir-03-tfidf.ipynb` hand-computes
TF/IDF instead of calling a library function.

## Frameworks used, and why

| Concept | Tool |
|---|---|
| Tokenization, lemmatization | **spaCy + scispaCy** (`en_core_sci_lg`) — already this project's pipeline for `10_kg_extraction.ipynb`, and biomedically aware (e.g. it does *not* split `T-cell` on the hyphen) |
| Stop words | spaCy's list vs. **NLTK**'s, compared directly |
| Stemming | **NLTK** — `PorterStemmer` (1980), `SnowballStemmer` ("Porter2", 2001), `LancasterStemmer` (Paice/Husk) — spaCy ships no stemmer by design, since it favors lemmatization |
| Accents/diacritics | stdlib `unicodedata` (NFKD), with **unidecode** for the cases NFKD can't handle |
| Modern subword tokenization | a from-scratch BPE trainer, then real **WordPiece** (BERT, BiomedBERT) and **SentencePiece** (T5) tokenizers via `transformers` |

`nltk` and `unidecode` are the only two packages this notebook adds to the
`kg-extraction` environment (see `environment/kg-extraction/requirements.txt`)
— everything else (`duckdb`, `spacy`, `scispacy`, `transformers`) is already
a dependency there, because this notebook is a diagnostic precursor to
`10_kg_extraction.ipynb`, not a separate lightweight stage.

## 1. Setup

Connects read-only to the DuckDB database built by `01_data_preparation.ipynb`,
loads the scispaCy pipeline already used elsewhere in this project, and
points NLTK's and Hugging Face's downloaders at `data/` (gitignored) instead
of the user's home directory, so every download this notebook makes stays
inside the project like the parquet/duckdb files from Step 0.

In [1]:
import os
import re
import unicodedata
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import duckdb
import pandas as pd
import spacy

warnings.filterwarnings("ignore", category=FutureWarning)  # spaCy tokenizer deserialization

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
DB_PATH = DATA_DIR / "clinical_cases.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found — run 01_data_preparation.ipynb first."
    )

# Keep NLTK data and Hugging Face model/tokenizer caches inside the gitignored
# data/ directory rather than the user's home directory.
NLTK_DATA_DIR = DATA_DIR / "nltk_data"
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(DATA_DIR / "hf_cache")

import nltk
nltk.data.path.insert(0, str(NLTK_DATA_DIR))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # NLTK warns about shared-machine download-dir permissions
    for pkg in ["stopwords"]:
        nltk.download(pkg, download_dir=str(NLTK_DATA_DIR), quiet=True)

con = duckdb.connect(str(DB_PATH), read_only=True)
nlp = spacy.load("en_core_sci_lg")
print("DB:", DB_PATH)
print("scispaCy pipeline:", nlp.pipe_names)

DB: /home/santanche/git/2learn/nlp2learn/to-kg/data/clinical_cases.duckdb
scispaCy pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


## 2. Normalization: the umbrella concept

> *Term* – a "normalized" word (case, morphology, spelling, etc.); an
> equivalence class of words. — Lecture 2, slide 16

Every section below (case folding, stemming, lemmatization, accent
stripping, ...) is really the same move: pick a function `normalize(word)`
and declare that any two words with the same output belong to one
**equivalence class**, i.e. one term. The classic example from the slides is
matching `U.S.A.` and `USA`. Here's that idea made literal — group a handful
of real author-name and abbreviation variants from this corpus by a
normalizer and see the equivalence classes fall out:

In [2]:
def normalize_loose(s: str) -> str:
    """Strip punctuation and lowercase — one possible normalize()."""
    return re.sub(r"[^a-z0-9]", "", s.lower())

variants = ["U.S.A.", "USA", "U.S.A", "usa", "CT", "C.T.", "ct", "MRI", "M.R.I."]

classes = defaultdict(list)
for v in variants:
    classes[normalize_loose(v)].append(v)

for term, members in classes.items():
    print(f"{term!r:6} <- {members}")

'usa'  <- ['U.S.A.', 'USA', 'U.S.A', 'usa']
'ct'   <- ['CT', 'C.T.', 'ct']
'mri'  <- ['MRI', 'M.R.I.']


## 3. Tokenization problem #1: hyphens — "one word or two?"

> Hewlett-Packard, state-of-the-art, co-education, data base,
> San Francisco-Los Angeles fares — Lecture 2, slide 21

First, what hyphenated terms actually dominate this corpus? (sampling
5,000 case texts rather than all 98,641 keeps this cell fast; the ranking is
stable across samples.)

In [3]:
sample = con.sql("SELECT case_text FROM cases USING SAMPLE 5000").fetchall()

hyphen_counts = Counter()
for (text,) in sample:
    for m in re.finditer(r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b", text):
        hyphen_counts[m.group(0).lower()] += 1

pd.DataFrame(hyphen_counts.most_common(15), columns=["hyphenated term", "count"])

,hyphenated term,count
0,year-old,3794
1,follow-up,2202
2,x-ray,577
3,c-reactive,349
4,right-sided,236
5,contrast-enhanced,224
6,long-term,208
7,left-sided,190
8,post-operative,169
9,sars-cov,157


Now tokenize a few of these with two different spaCy tokenizers: a **generic**
blank English tokenizer (`spacy.blank("en")`, default punctuation rules) and
the **biomedical** scispaCy tokenizer already loaded above
(`en_core_sci_lg`). Same input, same library, different rule set — which is
exactly the "is this one word or two" design decision the slide asks about,
made concrete.

In [4]:
nlp_generic = spacy.blank("en")

hyphen_examples = [
    "66-year-old", "contrast-enhanced", "T-cell lymphoma", "follow-up",
    "non-Hodgkin lymphoma", "stage-IV disease", "well-defined", "state-of-the-art",
]

rows = []
for ex in hyphen_examples:
    rows.append({
        "text": ex,
        "generic spaCy tokens": [t.text for t in nlp_generic(ex)],
        "scispaCy tokens": [t.text for t in nlp(ex)],
    })

pd.DataFrame(rows)

,text,generic spaCy tokens,scispaCy tokens
0,66-year-old,"[66, -, year, -, old]",[66-year-old]
1,contrast-enhanced,"[contrast, -, enhanced]",[contrast-enhanced]
2,T-cell lymphoma,"[T, -, cell, lymphoma]","[T-cell, lymphoma]"
3,follow-up,"[follow, -, up]",[follow-up]
4,non-Hodgkin lymphoma,"[non, -, Hodgkin, lymphoma]","[non-Hodgkin, lymphoma]"
5,stage-IV disease,"[stage, -, IV, disease]","[stage-IV, disease]"
6,well-defined,"[well, -, defined]",[well-defined]
7,state-of-the-art,"[state, -, of, -, the, -, art]",[state-of-the-art]


The generic tokenizer splits every hyphen into three tokens
(`66`, `-`, `year`, `-`, `old`) — the default infix rule treats `-` between
letters as a token boundary, same as the slide's ambiguity. scispaCy
overrides that rule and keeps the whole compound as one token, because in
biomedical text hyphens usually carry morphology, not word separation
(`T-cell`, `HER2-positive`, gene names) — splitting them would break
downstream NER. Neither choice is "correct" in the abstract; the slide's
point is exactly that this is a design decision, and here it's one you can
see the KG-extraction pipeline (`10_kg_extraction.ipynb`) has already made
for you by using scispaCy rather than a generic tokenizer.

## 4. Tokenization problem #2: "no whitespace" languages (aside)

> The two characters can be treated as one word meaning 'monk' or as a
> sequence of two words meaning 'and' and 'still'. — Lecture 2, slide 24
> (Chinese 和尚)

This corpus is entirely English clinical text, so there's no real instance of
this problem to mine from `cases`/`metadata` — pulling one in would mean
importing a segmenter (`jieba` for Chinese, MeCab/Sudachi for Japanese, or
spaCy's `zh_core_web_sm`/`ja_core_news_sm` pipelines) for a single aside, which
isn't worth the dependency here. Instead, the mechanism itself is easy to
reproduce directly: a minimal **forward maximum-match (FMM)** segmenter —
greedily consume the longest dictionary-known prefix — run over the slide's
own example with two different toy dictionaries reproduces both readings
mechanically, which is really the whole point of segmentation ambiguity: the
*algorithm* doesn't change, only the *dictionary* does.

In [5]:
def forward_max_match(text: str, dictionary: dict, max_len: int | None = None) -> list[str]:
    if max_len is None:
        max_len = max(len(w) for w in dictionary)
    tokens, i = [], 0
    while i < len(text):
        match = None
        for length in range(min(max_len, len(text) - i), 0, -1):
            candidate = text[i:i + length]
            if candidate in dictionary:
                match = candidate
                break
        match = match or text[i]  # fall back to a single character if nothing matched
        tokens.append(match)
        i += len(match)
    return tokens

text = "和尚"  # from slide 24
dict_monk = {"和尚": "monk"}
dict_and_still = {"和": "and", "尚": "still"}

print("dictionary knows '和尚' as one word:", forward_max_match(text, dict_monk))
print("dictionary only knows '和'/'尚' separately:", forward_max_match(text, dict_and_still))

dictionary knows '和尚' as one word: ['和尚']
dictionary only knows '和'/'尚' separately: ['和', '尚']


## 5. Accents and diacritics

> résumé vs. resume; Universität vs. Universitaet — Lecture 2, slide 29

`metadata.authors` (76,137 articles, several authors each) is a large,
real source of accented Latin names — a much better source than a
contrived example. Find distinct author names containing any non-ASCII
character:

In [6]:
authors = con.sql("SELECT DISTINCT UNNEST(authors) AS author FROM metadata").fetchall()

def has_non_ascii(s: str) -> bool:
    return any(ord(ch) > 127 for ch in s)

accented_authors = sorted({a for (a,) in authors if a and has_non_ascii(a)})
print(f"{len(accented_authors):,} of {len(authors):,} distinct author strings contain a non-ASCII character")
accented_authors[:10]

17,847 of 301,093 distinct author strings contain a non-ASCII character


['A A J Grüter',
 'A Antón',
 'A Armijo-Sánchez',
 'A B Giannì',
 'A Bañuelos-Franco',
 'A Blödow',
 'A Bucșa',
 'A Burián',
 'A C Maurício',
 'A Caliò']

The standard stdlib approach is Unicode **NFKD normalization**: it
decomposes each accented character into a base letter plus a separate
*combining* mark, then you drop anything `unicodedata.combining()` flags as
a mark. Shown character-by-character for one real name, so the
decomposition itself is visible rather than just its result:

In [7]:
def strip_accents(s: str) -> str:
    decomposed = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch))

example = "Straticò"
decomposed = unicodedata.normalize("NFKD", example)
print(f"{'original':10} {example}")
print(f"{'decomposed':10} {[ (c, unicodedata.name(c, '?')) for c in decomposed ]}")
print(f"{'stripped':10} {strip_accents(example)}")

original   Straticò
decomposed [('S', 'LATIN CAPITAL LETTER S'), ('t', 'LATIN SMALL LETTER T'), ('r', 'LATIN SMALL LETTER R'), ('a', 'LATIN SMALL LETTER A'), ('t', 'LATIN SMALL LETTER T'), ('i', 'LATIN SMALL LETTER I'), ('c', 'LATIN SMALL LETTER C'), ('o', 'LATIN SMALL LETTER O'), ('̀', 'COMBINING GRAVE ACCENT')]
stripped   Stratico


In [8]:
import random
random.seed(0)
sample_names = random.sample(accented_authors, 15)

pd.DataFrame({
    "original": sample_names,
    "NFKD-stripped": [strip_accents(n) for n in sample_names],
})

,original,NFKD-stripped
0,Miłosz Nesterowicz,Miłosz Nesterowicz
1,Patricia Ramírez de la Piscina,Patricia Ramirez de la Piscina
2,Andrés Ramos,Andres Ramos
3,José Mário Bastos,Jose Mario Bastos
4,Vera Maria Rêgo Durão,Vera Maria Rego Durao
5,Séamus Hussey,Seamus Hussey
6,Nicolás Coronel-Restrepo,Nicolas Coronel-Restrepo
7,Laura Segura-Muñoz,Laura Segura-Munoz
8,Smaïl El Youssoufi,Smail El Youssoufi
9,María Esnaola Azcoiti,Maria Esnaola Azcoiti


**A real gotcha, found in this corpus, not invented for the slide:** NFKD
only helps when the accented character *decomposes* into base + combining
mark. Some letters — Polish `Ł` (L with stroke), Turkish dotless `ı`,
Danish `ø` — are their own distinct Unicode code points with no such
decomposition, so `strip_accents` silently leaves them untouched:

In [9]:
from unidecode import unidecode

tricky = "Monika MaŁgorzata Adamska"  # real author string in metadata.authors
print("NFKD strip :", strip_accents(tricky), " <- unchanged, Ł has no combining-mark decomposition")
print("unidecode  :", unidecode(tricky), " <- transliteration table handles it")

NFKD strip : Monika MaŁgorzata Adamska  <- unchanged, Ł has no combining-mark decomposition
unidecode  : Monika MaLgorzata Adamska  <- transliteration table handles it


## 6. Case folding

> MIT vs. mit; Fed vs. fed — Lecture 2, slide 31

Clinical text is full of exactly this ambiguity: short, all-caps
abbreviations that collide with ordinary English words once lowercased.
Pull real sentences from `cases.case_text` containing four such
abbreviations (`US`, `OR`, `AS`, `IS`), with enough surrounding context to
see the clinical meaning next to the everyday word it folds into:

In [10]:
targets = {
    "US": "ultrasound/ultrasonography",
    "OR": "operating room",
    "AS": "aortic stenosis / ankylosing spondylitis (context-dependent)",
    "IS": "in situ",
}

sample = con.sql("SELECT case_text FROM cases USING SAMPLE 8000").fetchall()

rows = []
for abbrev, meaning in targets.items():
    for (text,) in sample:
        m = re.search(rf"\b{abbrev}\b", text)
        if m:
            start = max(0, m.start() - 45)
            context = text[start:m.end() + 25].replace("\n", " ")
            rows.append({
                "abbreviation": abbrev,
                "clinical meaning": meaning,
                "lowercased to": abbrev.lower(),
                "collides with common word": f"'{abbrev.lower()}'",
                "context in corpus": f"...{context}...",
            })
            break

pd.DataFrame(rows)

,abbreviation,clinical meaning,lowercased to,collides with common word,context in corpus
0,US,ultrasound/ultrasonography,us,'us',...ient boarded a plane from the Bahamas to th...
1,OR,operating room,or,'or',...ure of the BPF. The patient was taken to th...
2,AS,aortic stenosis / ankylosing spondylitis (cont...,as,'as',...ld male security guard. H was diagnosed wit...
3,IS,in situ,is,'is',...area. OCT imagery revealed disruption of th...


A naive `.lower()` fold would merge `US` (ultrasound) into the same term as
every occurrence of the pronoun "us" — exactly the false-merge the slide
warns about with `MIT`/`mit`. One real mitigation: fold everything *except*
tokens on a curated abbreviation allowlist, and only when they're not at the
start of a sentence (where capitalization is just orthography, not
meaning):

In [11]:
CLINICAL_ABBREVIATIONS = {"US", "OR", "AS", "IS", "CT", "MRI", "ICU", "CRP", "WBC"}

def smart_lower(token: str, sentence_initial: bool = False) -> str:
    if token in CLINICAL_ABBREVIATIONS and not sentence_initial:
        return token  # preserve — folding would collide with a common word
    return token.lower()

for tok, sentence_initial in [("US", False), ("US", True), ("Patient", True), ("OR", False)]:
    print(f"smart_lower({tok!r}, sentence_initial={sentence_initial}) -> {smart_lower(tok, sentence_initial)!r}")

smart_lower('US', sentence_initial=False) -> 'US'
smart_lower('US', sentence_initial=True) -> 'us'
smart_lower('Patient', sentence_initial=True) -> 'patient'
smart_lower('OR', sentence_initial=False) -> 'OR'


## 7. Stop words

> stop words = extremely common words which would appear to be of little
> value in helping select documents ... but you need stop words for phrase
> queries, e.g. "King of Denmark" — Lecture 2, slide 32

Two off-the-shelf lists, compared directly rather than taken on faith:

In [12]:
from nltk.corpus import stopwords as nltk_stopwords

spacy_stop = nlp.Defaults.stop_words
nltk_stop = set(nltk_stopwords.words("english"))

print(f"spaCy/scispaCy stop list: {len(spacy_stop)} words")
print(f"NLTK stop list:           {len(nltk_stop)} words")
print()
print("in spaCy but not NLTK (sample):", sorted(spacy_stop - nltk_stop)[:10])
print("in NLTK but not spaCy (sample):", sorted(nltk_stop - spacy_stop)[:10])

spaCy/scispaCy stop list: 326 words
NLTK stop list:           198 words

in spaCy but not NLTK (sample): ["'d", "'ll", "'m", "'re", "'s", "'ve", 'across', 'afterwards', 'almost', 'alone']
in NLTK but not spaCy (sample): ['ain', 'aren', "aren't", 'couldn', "couldn't", 'd', 'didn', "didn't", 'doesn', "doesn't"]


Applied to a real case sentence — this is the sentence reused in the
lemmatization and stemming sections below, so effects on the *same* text
are comparable across sections:

In [13]:
PIPELINE_SENTENCE = (
    "A 66-year-old female with a history of endometrioid endometrial carcinoma "
    "was sent to our institution with abdominal and pelvic pain."
)  # PMC6180851_01, opening sentence

doc = nlp(PIPELINE_SENTENCE)
tokens = [t.text for t in doc]
kept = [t.text for t in doc if not t.is_stop and not t.is_punct]

print("all tokens :", tokens)
print("with stop words removed:", kept)

all tokens : ['A', '66-year-old', 'female', 'with', 'a', 'history', 'of', 'endometrioid', 'endometrial', 'carcinoma', 'was', 'sent', 'to', 'our', 'institution', 'with', 'abdominal', 'and', 'pelvic', 'pain', '.']
with stop words removed: ['66-year-old', 'female', 'history', 'endometrioid', 'endometrial', 'carcinoma', 'sent', 'institution', 'abdominal', 'pelvic', 'pain']


Slide 32's warning is visible right here: removing stop words from
`"history of endometrioid endometrial carcinoma"` loses the word "of" — fine
for a bag-of-words match, but it would break an exact **phrase query** for
that span, which is exactly why most modern search engines (and the
positional/biword indexes from the rest of this lecture) keep stop words
indexed rather than discarding them.

## 8. Lemmatization

> am, are, is → be; car, cars, car's, cars' → car — Lecture 2, slide 34

scispaCy's lemmatizer is rule-based over POS tags (`nlp.get_pipe("lemmatizer").mode`
below), not a lookup table, so it needs the sentence's part-of-speech tags to
decide, e.g., whether `was` is a verb (→ `be`) — this is why lemmatization
needs a full pipeline and stemming (next section) doesn't.

In [14]:
print("lemmatizer mode:", nlp.get_pipe("lemmatizer").mode)

doc = nlp(PIPELINE_SENTENCE)
pd.DataFrame([
    {"token": t.text, "lemma": t.lemma_, "POS": t.pos_, "tag": t.tag_}
    for t in doc
])

lemmatizer mode: rule


,token,lemma,POS,tag
0,A,a,DET,DT
1,66-year-old,66-year-old,ADJ,JJ
2,female,female,NOUN,NN
3,with,with,ADP,IN
4,a,a,DET,DT
5,history,history,NOUN,NN
6,of,of,ADP,IN
7,endometrioid,endometrioid,ADJ,JJ
8,endometrial,endometrial,ADJ,JJ
9,carcinoma,carcinoma,NOUN,NN


Worth flagging honestly: rule-based lemmatization isn't perfect on domain
vocabulary. Run it on a plural clinical noun and see:

In [15]:
doc2 = nlp("The scan revealed multiple masses.")
for t in doc2:
    if t.text == "masses":
        print(f"{t.text!r} -> lemma {t.lemma_!r}  (expected 'mass' — 'masse' is a mis-lemmatization)")

'masses' -> lemma 'masse'  (expected 'mass' — 'masse' is a mis-lemmatization)


## 9. Stemming

> Crude heuristic process that chops off the ends of words in the hope of
> achieving what "principled" lemmatization attempts to do ...
> — Lecture 2, slide 35

### 9.1 The Porter rule table, by hand

Slide 37 shows four of Porter's Step-1a suffix rules directly. They're
simple enough to implement literally and check against the slide's own
examples before reaching for a library:

In [16]:
def porter_step1a(word: str) -> str:
    if word.endswith("sses"):
        return word[:-2]          # SSES -> SS      caresses -> caress
    if word.endswith("ies"):
        return word[:-2]          # IES -> I        ponies -> poni
    if word.endswith("ss"):
        return word               # SS -> SS        caress -> caress
    if word.endswith("s"):
        return word[:-1]          # S -> (nothing)  cats -> cat
    return word

for w in ["caresses", "ponies", "caress", "cats"]:
    print(f"{w:12} -> {porter_step1a(w)}")

caresses     -> caress
ponies       -> poni
caress       -> caress
cats         -> cat


This is only Step 1a of five phases in the real algorithm (see slide 36) —
enough to see the *shape* of the rules, not enough to stem real vocabulary
correctly (e.g. it would turn `analysis` into `analysi`, wrongly, because
later phases that fix that haven't been implemented here). For anything
beyond a teaching example, use a maintained implementation.

### 9.2 Is Porter (1980) the most recent? No.

Porter revised his own algorithm around 2001 to fix edge cases in the
original — usually called **Porter2** or the "English Snowball" stemmer.
NLTK ships both: `PorterStemmer` is the original 1980 algorithm (kept for
reproducibility), `SnowballStemmer("english")` is the 2001 revision. A third,
more aggressive stemmer from the same era, **Lancaster** (Paice/Husk,
1990), is the third stemmer slide 38 itself compares against Porter and
Lovins — NLTK ships it as `LancasterStemmer`.

In [17]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem.lancaster import LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

words = [
    "caresses", "ponies",                                            # slide 37
    "operate", "operating", "operates", "operation", "operative", "operational",  # slide 39
]

pd.DataFrame([
    {"word": w, "Porter (1980)": porter.stem(w), "Snowball / Porter2 (2001)": snowball.stem(w), "Lancaster (Paice)": lancaster.stem(w)}
    for w in words
])

,word,Porter (1980),Snowball / Porter2 (2001),Lancaster (Paice)
0,caresses,caress,caress,caress
1,ponies,poni,poni,pony
2,operate,oper,oper,op
3,operating,oper,oper,op
4,operates,oper,oper,op
5,operation,oper,oper,op
6,operative,oper,oper,op
7,operational,oper,oper,op


Slide 39's `oper` equivalence class (operate/operating/operates/operation/
operative/operational, all collapsing to one stem) is a curated example.
Does the same pattern hold for real clinical morphology in this corpus? Stem
several word families actually present in `cases.case_text`, with their real
corpus frequency (from an 8,000-row sample) attached:

In [18]:
word_families = {
    "diagnosis": 4772, "diagnosed": 2764, "diagnostic": 762, "diagnoses": 200, "diagnose": 46,
    "metastasis": 833, "metastatic": 796, "metastases": 652, "metastasized": 11,
    "biopsy": 2864, "biopsies": 365, "biopsied": 61,
}

pd.DataFrame([
    {
        "word": w, "corpus freq": freq,
        "Porter": porter.stem(w), "Snowball": snowball.stem(w), "Lancaster": lancaster.stem(w),
    }
    for w, freq in word_families.items()
])

,word,corpus freq,Porter,Snowball,Lancaster
0,diagnosis,4772,diagnosi,diagnosi,diagnos
1,diagnosed,2764,diagnos,diagnos,diagnos
2,diagnostic,762,diagnost,diagnost,diagnost
3,diagnoses,200,diagnos,diagnos,diagnos
4,diagnose,46,diagnos,diagnos,diagnos
5,metastasis,833,metastasi,metastasi,metastas
6,metastatic,796,metastat,metastat,metast
7,metastases,652,metastas,metastas,metastas
8,metastasized,11,metastas,metastas,metastas
9,biopsy,2864,biopsi,biopsi,biopsy


Slide 39 also warns that stemming can *hurt*: it names the family
`operational AND research` as one where merging equivalence classes creates
false matches. The table above shows the opposite failure mode happening for
real, un-curated corpus vocabulary: **`metastasis` stems to `metastasi`,
but its own plural `metastases` stems to `metastas`** under both Porter and
Snowball — two forms of the same word land in *different* equivalence
classes, exactly the kind of silent miss stemming is supposed to prevent.
(Lancaster's more aggressive rules happen to unify this particular pair, at
the cost of being wrong in other ways — no stemmer choice is free of
trade-offs.) Likewise `diagnosis`/`diagnosed`/`diagnostic` land on three
different Porter/Snowball stems despite being the same clinical concept.

## 10. Modern practice: subword tokenization (BPE / WordPiece / SentencePiece)

Neither stemming nor lemmatization survived into the transformer era.
Instead, every model this activity's later notebooks depend on
(`10_kg_extraction.ipynb` uses `transformers`/`torch` encoders) is trained
over a fixed, **data-driven subword vocabulary**: instead of a linguist's
rules for what a "word" or "root" is, a training corpus is scanned for
statistically frequent character sequences, and words are split into
whatever pieces of that learned vocabulary reconstruct them. This sidesteps
hyphenation, compounding, and unseen morphology all at once, at the cost of
losing the clean "one token = one meaning-bearing lemma" property stemming
and lemmatization aimed for.

Three related but distinct algorithms dominate:

| Algorithm | Merge/split criterion | Used by |
|---|---|---|
| **BPE** (Sennrich et al., 2016) | greedily merge the *most frequent* adjacent symbol pair | GPT-family models |
| **WordPiece** (Schuster & Nakajima, 2012) | BPE-like, but merges by *likelihood gain* rather than raw frequency | BERT family |
| **SentencePiece / Unigram** (Kudo, 2018) | start from a large vocabulary and *prune* low-probability pieces under a unigram language model | T5, ALBERT, XLNet |

### 10.1 BPE, by hand — same style as the TF-IDF notebook

BPE's training loop is short enough to implement directly, the same way
`ir-03-tfidf.ipynb` hand-computes TF and IDF instead of calling a library.
Train it on the real per-word frequencies of the `diagnos-`/`metasta-`/
`biops-`/`endometri-`/`carcinom-` families counted from an 8,000-row corpus
sample — a real, if small, training corpus rather than a toy one:

In [19]:
sample = con.sql("SELECT case_text FROM cases USING SAMPLE 8000").fetchall()

roots = ["biops", "metasta", "endometri", "diagnos", "carcinom"]
word_freqs = Counter()
for (text,) in sample:
    for m in re.finditer(r"\b[a-zA-Z]+\b", text):
        w = m.group(0).lower()
        if any(w.startswith(r) for r in roots):
            word_freqs[w] += 1

BPE_TRAIN_WORDS = dict(word_freqs.most_common(23))  # drop the long tail of 1-2 occurrence typos
pd.DataFrame(BPE_TRAIN_WORDS.items(), columns=["word", "corpus freq"])

,word,corpus freq
0,diagnosis,4617
1,biopsy,3000
2,diagnosed,2668
3,carcinoma,1026
4,metastasis,847
5,metastases,736
6,metastatic,727
7,diagnostic,716
8,biopsies,323
9,diagnoses,172


In [20]:
EOW = "</w>"  # end-of-word marker, so e.g. the "s" in "biops" is distinguishable from a word-final "s"

def word_to_symbols(word: str) -> list[str]:
    return list(word) + [EOW]

def get_pair_stats(vocab: dict) -> dict:
    pairs = defaultdict(int)
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, vocab: dict) -> dict:
    bigram = "".join(pair)
    new_vocab = {}
    for symbols, freq in vocab.items():
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                merged.append(bigram)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        key = tuple(merged)
        new_vocab[key] = new_vocab.get(key, 0) + freq
    return new_vocab

vocab = {tuple(word_to_symbols(w)): f for w, f in BPE_TRAIN_WORDS.items()}
merges, merge_log = [], []

N_MERGES = 15
for step in range(N_MERGES):
    pairs = get_pair_stats(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    merge_log.append({"step": step + 1, "pair": best_pair, "merged into": "".join(best_pair), "pair freq": pairs[best_pair]})
    vocab = merge_pair(best_pair, vocab)
    merges.append(best_pair)

pd.DataFrame(merge_log)

,step,pair,merged into,pair freq
0,1,"(n, o)",no,9405
1,2,"(i, a)",ia,8438
2,3,"(d, ia)",dia,8315
3,4,"(dia, g)",diag,8315
4,5,"(diag, no)",diagno,8315
5,6,"(diagno, s)",diagnos,8315
6,7,"(s, </w>)",s</w>,6868
7,8,"(i, s</w>)",is</w>,5535
8,9,"(t, a)",ta,4666
9,10,"(diagnos, is</w>)",diagnosis</w>,4617


Applying the 15 learned merges to words the trainer never saw shows the
payoff: BPE reuses subwords it already learned (`biops`, `tas`, `diagnos`)
to cover unseen inflections, instead of failing outright the way a
whitespace/hyphen tokenizer would on a genuinely novel string:

In [21]:
def bpe_tokenize(word: str, merges: list[tuple[str, str]]) -> list[str]:
    symbols = word_to_symbols(word)
    for pair in merges:
        bigram = "".join(pair)
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                merged.append(bigram)
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        symbols = merged
    return symbols

for w in ["metastasizing", "endometriotic", "carcinomatous", "biopsying"]:  # none were in BPE_TRAIN_WORDS
    print(f"{w:16} -> {bpe_tokenize(w, merges)}")

metastasizing    -> ['m', 'e', 'tas', 'tas', 'i', 'z', 'i', 'n', 'g', '</w>']
endometriotic    -> ['e', 'n', 'd', 'o', 'm', 'e', 't', 'r', 'io', 't', 'i', 'c', '</w>']
carcinomatous    -> ['c', 'a', 'r', 'c', 'i', 'no', 'm', 'a', 't', 'o', 'u', 's</w>']
biopsying        -> ['biops', 'y', 'i', 'n', 'g', '</w>']


### 10.2 Real pretrained tokenizers: WordPiece and SentencePiece

The from-scratch version above trains in seconds on 23 words; production
tokenizers train the same kind of loop on billions of words. Load three
real ones — generic-English WordPiece (BERT), biomedical WordPiece
(BiomedBERT, trained on PubMed abstracts/full text), and SentencePiece
(T5) — and run them on the pipeline sentence used throughout this notebook.
(`##` marks a WordPiece continuation of the previous token; `▁` marks a
SentencePiece token that starts a new word.)

In [22]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
biomed_tok = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
t5_tok = AutoTokenizer.from_pretrained("t5-small")

tokenizers = {
    "BERT (generic WordPiece)": bert_tok,
    "BiomedBERT (domain WordPiece)": biomed_tok,
    "T5 (SentencePiece/Unigram)": t5_tok,
}

for name, tok in tokenizers.items():
    pieces = tok.tokenize(PIPELINE_SENTENCE)
    print(f"{name:32} {len(pieces):3} tokens: {pieces}")

/home/santanche/git/2learn/nlp2learn/to-kg/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERT (generic WordPiece)          35 tokens: ['a', '66', '-', 'year', '-', 'old', 'female', 'with', 'a', 'history', 'of', 'end', '##ome', '##tri', '##oid', 'end', '##ome', '##tri', '##al', 'car', '##cino', '##ma', 'was', 'sent', 'to', 'our', 'institution', 'with', 'abdominal', 'and', 'pe', '##l', '##vic', 'pain', '.']
BiomedBERT (domain WordPiece)     27 tokens: ['a', '66', '-', 'year', '-', 'old', 'female', 'with', 'a', 'history', 'of', 'endometr', '##io', '##id', 'endometrial', 'carcinoma', 'was', 'sent', 'to', 'our', 'institution', 'with', 'abdominal', 'and', 'pelvic', 'pain', '.']
T5 (SentencePiece/Unigram)        36 tokens: ['▁A', '▁', '66', '-', 'year', '-', 'old', '▁female', '▁with', '▁', 'a', '▁history', '▁of', '▁end', 'o', 'metri', 'o', 'i', 'd', '▁end', 'o', 'metri', 'al', '▁carcinoma', '▁was', '▁sent', '▁to', '▁our', '▁institution', '▁with', '▁abdominal', '▁and', '▁pelvi', 'c', '▁pain', '.']


Generic BERT fragments `endometrioid` and `carcinoma` into four and three
pieces respectively, because those words are rare in its general-web/book
training data. BiomedBERT, trained on biomedical text, keeps
`endometrial`/`carcinoma` whole and needs noticeably fewer tokens overall
for the same sentence — the subword-era version of exactly the same
domain-vocabulary argument that motivates using `en_core_sci_lg` instead of
a generic spaCy model in section 3, and using scispaCy/biomedical models in
`10_kg_extraction.ipynb`.

### 10.3 Everything on one sentence

Every tokenization/normalization strategy from this notebook, applied to
the words of `PIPELINE_SENTENCE`, side by side:

In [23]:
words_in_sentence = [t.text for t in nlp(PIPELINE_SENTENCE) if not t.is_punct]

pd.DataFrame([
    {
        "word": w,
        "scispaCy lemma": nlp(w)[0].lemma_,
        "Porter stem": porter.stem(w),
        "Snowball stem": snowball.stem(w),
        "hand-rolled BPE": "".join(bpe_tokenize(w.lower(), merges)).replace(EOW, ""),
        "BERT WordPiece": "|".join(bert_tok.tokenize(w)),
    }
    for w in words_in_sentence
])

,word,scispaCy lemma,Porter stem,Snowball stem,hand-rolled BPE,BERT WordPiece
0,A,a,a,a,a,a
1,66-year-old,66-year-old,66-year-old,66-year-old,66-year-old,66|-|year|-|old
2,female,female,femal,femal,female,female
3,with,with,with,with,with,with
4,a,a,a,a,a,a
5,history,history,histori,histori,history,history
6,of,of,of,of,of,of
7,endometrioid,endometrioid,endometrioid,endometrioid,endometrioid,end|##ome|##tri|##oid
8,endometrial,endometrial,endometri,endometri,endometrial,end|##ome|##tri|##al
9,carcinoma,carcinoma,carcinoma,carcinoma,carcinoma,car|##cino|##ma


## 11. Take-away

| Slide concept | Tool used here | Grounded in |
|---|---|---|
| Normalization (equivalence classes) | hand-rolled `normalize()` | U.S.A./USA/CT/C.T. variants |
| Hyphen tokenization | spaCy blank vs. scispaCy | `T-cell`, `follow-up`, `contrast-enhanced` (real corpus counts) |
| "No whitespace" languages | hand-rolled forward-max-match | 和尚 (slide's own example — out of corpus scope) |
| Accents/diacritics | `unicodedata` NFKD, `unidecode` | 17,847 accented author names in `metadata.authors` |
| Case folding | curated allowlist fold | `US`/`OR`/`AS`/`IS` vs. common-word collisions, real sentences |
| Stop words | spaCy vs. NLTK list diff | `PIPELINE_SENTENCE` |
| Lemmatization | scispaCy rule-based lemmatizer | same sentence + a mis-lemmatization caveat |
| Stemming | Porter / Snowball (Porter2) / Lancaster | `diagnos-`/`metasta-`/`biops-` word families |
| Modern subwording | hand-rolled BPE, BERT/BiomedBERT WordPiece, T5 SentencePiece | same word families + `PIPELINE_SENTENCE` |

The working pipeline this project actually uses downstream
(`10_kg_extraction.ipynb`) is scispaCy (`en_core_sci_lg`) for
tokenization/lemmatization/NER, plus transformer encoders with their own
WordPiece/SentencePiece vocabularies — informed by, but not identical to,
every technique explored above; this notebook is the diagnostic tour, not a
new pipeline.

In [24]:
con.close()